# 02. 1/N against sample-based mean-variance: the rolling out-of-sample test

**Terms used in this notebook.**

| Term | Meaning |
|---|---|
| Risk-free rate | the return on a one-month US Treasury bill, the closest thing to a return with no risk |
| Excess return | a return minus the risk-free rate over the same month |
| Sharpe ratio | the average monthly excess return divided by the standard deviation of the excess return; the reward earned per unit of risk taken |
| Certainty-equivalent return (CEQ) | the average return minus half the variance (times a risk aversion of one); the sure monthly return an investor would accept in place of the risky one |
| Turnover | the fraction of the portfolio bought and sold at a rebalance, summed over the assets; a turnover of 0.02 means that 2% of the portfolio changes hands in the month |
| Estimation window | the M months of past returns a rule sees when it forms its weights, 120 here |
| Rolling evaluation | moving the window forward one month at a time, forming the weights, and recording the return of the month after the window |
| Out-of-sample | measured on months the rule had not seen when it formed its weights |
| In-sample | measured on the same months that were used to form the weights |
| Estimation error | the difference between a mean or covariance estimated from a window and its true value |
| Band | the interval, fixed before any code ran, within which a replicated number counts as matching the published one, 0.03 for a Sharpe ratio and 25% for turnover |
| Gated | said of a cell the band applies to |
| Verdict | pass or MISS for a gated cell |
| Fragility test | adding small random noise to every return and rerunning a rule fifty times, to see how far its result can move on a slightly different version of the data |
| Basis point | one hundredth of a percentage point, so 10 basis points is 0.1% |
| Vintage | the version of Ken French's files on the download date, named by the release of the CRSP database they were built from (202607 is the July 2026 release) |
| Provenance | the record of what was downloaded, when, with its checksum and vintage |
| Factor | a return series that moves many assets at once, for example the return of the whole market; the Fama-French factors SMB and HML are the returns of small firms over large ones and of cheap firms over expensive ones |
| Market capitalisation | the number of a firm's shares times their price, the firm's size in money |
| Value-weighted return | the return of a group of firms in which each firm counts in proportion to its market capitalisation at the start of the month |
| Upper bound | the Sharpe ratio a mean-variance investor would earn with no estimation error, from a rule estimated on the whole sample and judged on the same sample |
| Simulated history | the 24,000 months of returns the simulation produces |
| Seed | the number that fixes the random numbers a simulation uses; the same seed gives the same random numbers |
| Draw | one simulated history, produced from one seed |
| Standard error | the uncertainty of an estimate; for an average over M months it is the standard deviation of the months divided by the square root of M |
| P-value | the probability of a difference at least as large as the one observed if the two rules were equally good; below 0.05 the difference is unlikely to be chance |
| Jobson-Korkie | the standard test of whether two Sharpe ratios measured on the same months differ; its result is reported as a p-value |
| Delta method | a standard way to obtain a standard error for a quantity built from means and variances, used for the CEQ |
| Covariance matrix | the table of all variances and covariances of a set of assets; the covariance of two assets says how they move together, positive when both tend to be above their averages in the same months |
| Near-singular | said of a covariance matrix in which one asset's return is almost a combination of the others', so that the matrix carries almost no information about the difference between them; inverting it then divides by a number close to zero |

**What this notebook does.** It builds the four DGU datasets from French's files, runs the 1/N rule and the sample-based mean-variance rule through DGU's rolling 120-month window, and puts the out-of-sample Sharpe ratio, certainty-equivalent return and turnover beside Tables 3, 4 and 5 of the paper, with the verdict of the band I wrote down before the run.

**Why it comes second.** These two rules are the paper's two extremes. 1/N estimates nothing. Sample-based mean-variance estimates everything, a mean and a covariance for every asset, and plugs the estimates into Markowitz's formula as if they were true. Every other rule in the paper is an attempt to move the second toward the first, so this pair has to be right before any of them is worth running.

**The method.** At the end of each month t, from t = 120 onward, the rule sees the previous 120 months of excess returns, forms its weights, and holds them through month t + 1. Over 497 months that is 377 out-of-sample returns per rule and dataset (DGU section 2). The Sharpe ratio is the mean of those 377 returns over their standard deviation, equation (12); the CEQ return is the mean less half the variance, equation (14) with gamma = 1; turnover is the average absolute trade needed each month to move from the weights the last month's returns drifted the portfolio to and the new target, equation (15). The mean-variance weights are Sigma^-1 mu, equation (3), scaled so that they sum to one in absolute value, equation (1). The 1/N weights are 1/N.

**What I expect to see, and where it comes from.** From Table 3, monthly Sharpe ratios for 1/N of 0.1353, 0.2240, 0.1623 and 0.1753 on Industry, MKT/SMB/HML, FF-1-factor and FF-4-factor; for mean-variance out of sample 0.0679, 0.2186, 0.0128 and 0.1841; for mean-variance in sample, the upper bound estimation error costs, 0.2124, 0.2851, 0.5098 and 0.5364. From Table 5, 1/N turns over 2.2%, 2.4%, 1.6% and 2.0% of the portfolio a month, and mean-variance 606,594, 2.83, 10,466 and 3,553 times that. The band written down before the run: an out-of-sample Sharpe ratio within 0.03 of the published one passes, and turnover within 25% where the published relative turnover is below 100; above that the number is reported and not gated. The value-weighted market, "vw", is run too, as the data check from notebook 01.

**One amendment, made after the first run and dated.** On 13 September 2026, on the evidence of the fragility test at the end of this notebook, the Sharpe band was restricted to the same rows as the turnover band: rules whose published relative turnover is below 100. The unconstrained mean-variance rule on the Industry, FF-1-factor and FF-4-factor datasets is above that level and is reported with the range that noise produces, with no verdict; the in-sample row carries no verdict either, because the band was written for out-of-sample ratios. The level is the paper's own number; it was not chosen from these results. The amendment is recorded in `src/bp/constants.py` under `BAND_SHARPE_APPLIES_BELOW_RELATIVE`.

## Modules

The first four cells are the infrastructure from notebook 01 (the constants, the loader, DGU's published tables, and the dataset builder), unchanged. The three after them are the substance of this notebook: the rules, the rolling evaluation, and the comparison against the paper.

In [ ]:
import os
os.makedirs("src/bp", exist_ok=True)
open("src/bp/__init__.py", "w").close()

In [ ]:
%%writefile src/bp/constants.py
"""
Pre-committed numbers for the study.

Every band, gate, sample boundary and design parameter is defined here, one
per line, with its source beside it. A notebook may read these names; it may
not introduce numbers of its own. The replication bands (section 3) and the
design of the extension (section 5) were fixed on 10 September 2026, before any
of the rules' code was written. The expectations each notebook tests are
stated in that notebook's introduction and were written before its code; where
a level was changed after a run, the comment beside it says so, with the date
and the reason, so that a reader can judge whether the change was a
specification settled by the paper's own text or a number tuned to the results.

DGU throughout means DeMiguel, Garlappi and Uppal (2009), "Optimal Versus
Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?",
Review of Financial Studies 22(5), 1915-1953. Page numbers are the journal's.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":   "F-F_Research_Data_Factors_CSV.zip",        # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "momentum":   "F-F_Momentum_Factor_CSV.zip",              # Mom, monthly from 1927-01
    "factors5":   "F-F_Research_Data_5_Factors_2x3_CSV.zip",  # Mkt-RF, SMB, HML, RMW, CMA, RF from 1963-07
    "ind10":      "10_Industry_Portfolios_CSV.zip",           # DGU "Industry" dataset
    "ind49":      "49_Industry_Portfolios_CSV.zip",           # the extension universe; carries firm counts and average size
    "size_bm25":  "25_Portfolios_5x5_CSV.zip",                # DGU "FF-1-factor" and "FF-4-factor" datasets
    # Daily files, for the extension's covariance estimators (section 5).
    "ind49_daily":    "49_Industry_Portfolios_daily_CSV.zip",        # value- and equal-weighted daily returns from 1926-07-01
    "factors3_daily": "F-F_Research_Data_Factors_daily_CSV.zip",     # Mkt-RF, SMB, HML, RF daily from 1926-07-01
    "factors5_daily": "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip", # Mkt-RF, SMB, HML, RMW, CMA, RF daily from 1963-07-01
    "momentum_daily": "F-F_Momentum_Factor_daily_CSV.zip",           # Mom daily from 1926-11-03
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# ---------------------------------------------------------------------------
# 2. The DGU replication
# ---------------------------------------------------------------------------

DGU_SAMPLE_START = "1963-07"   # DGU Table 2, p. 1918, for every French-sourced dataset
DGU_SAMPLE_END = "2004-11"     # DGU Table 2, p. 1918
DGU_ESTIMATION_WINDOW = 120    # months; DGU section 2, p. 1927. M = 60 is in DGU's appendix and is out of scope.
DGU_RISK_AVERSION = 1.0        # gamma in the CEQ, DGU p. 1929: "the results we report are for the case of gamma = 1"
DGU_TCOST = 0.0050             # proportional cost per unit of one-way turnover; DGU p. 1929: 50 basis points, after Balduzzi and Lynch (1999)
TCOST_SENSITIVITY = 0.0100     # the one cost sensitivity reported: double the base cost
DGU_GMINC_LOWER_BOUND_FRACTION = 0.5   # g-min-c imposes w >= a * 1 with a = 1/(2N); DGU p. 1926
DGU_BS_COV_DOF = "M - N - 2"   # the Bayes-Stein covariance divides by M - N - 2; DGU equation (5), p. 1923

# Settled 22 September 2026, a specification the paper leaves ambiguous. DGU's
# Lagrangian (8), p. 1925, shows the short-sale constraint alone; footnote 22,
# p. 1934, says the constrained rules produce "corner solutions with all wealth
# invested in a single asset", which only happens when the budget constraint
# 1'w = 1 sits inside the optimisation with gamma = 1. With the budget inside,
# mv-c on MKT/SMB/HML gives a Sharpe ratio of 0.1093 against the published
# 0.1084 and a corner solution in 257 of 377 months; without it, 0.2382 and no
# corner. The reading that reproduces the paper's tables is used; the literal
# reading is kept as rules.mv_c_cone for comparison (notebook 03).
DGU_CONSTRAINED_BUDGET_INSIDE = True

# The rules replicated: those whose only inputs are the window's sample moments
# (and, for Bayes-Stein, Jorion's prior built from them), plus the value-weighted
# market, whose Sharpe ratio is the same 0.1138 in every French-sourced column
# of Table 3 and so checks that the market series and the sample match the paper's.
DGU_RULES = ("ew", "mv", "bs", "min", "vw", "mv-c", "bs-c", "min-c", "g-min-c")
DGU_IN_SAMPLE_ROW = "mv (in sample)"   # DGU Table 3 row 2: mean-variance with M = T, the upper bound estimation error costs

# Excluded, each with the criterion. They are not built.
DGU_RULES_EXCLUDED = {
    "Bayesian diffuse-prior": "not reported by DGU (Table 1 note, p. 1917)",
    "dm": "Bayesian Data-and-Model (Pastor 2000); needs an asset-pricing prior, outside the moment-based set",
    "mp": "MacKinlay and Pastor (2000) missing-factor model; needs a maximum-likelihood factor structure, outside the moment-based set",
    "mv-min": "Kan and Zhou (2007) three-fund mixture of two rules already built; not part of the pre-specified set",
    "ew-min": "mixture of 1/N and minimum variance; not part of the pre-specified set",
    "multi-prior": "not reported by DGU (Table 1 note, p. 1917)",
}

# The datasets replicated: the four DGU built from French's site (DGU Table 2, p. 1918; Appendix A, pp. 1948-1949).
# Each is monthly excess returns over the T-bill rate from French's factor file.
DGU_DATASETS = {
    "Industry":     {"n_risky": 10, "factors": ("Mkt-RF",), "N": 11, "source": "ind10 + factors3"},
    "MKT/SMB/HML":  {"n_risky": 0,  "factors": ("Mkt-RF", "SMB", "HML"), "N": 3, "source": "factors3"},
    "FF-1-factor":  {"n_risky": 20, "factors": ("Mkt-RF",), "N": 21, "source": "size_bm25 + factors3"},
    "FF-4-factor":  {"n_risky": 20, "factors": ("Mkt-RF", "SMB", "HML", "Mom"), "N": 24, "source": "size_bm25 + factors3 + momentum"},
}
DGU_DATASETS_EXCLUDED = {
    "S&P Sectors": "Roberto Wessels' GICS sector data, not on French's site (DGU Appendix A.1)",
    "International": "MSCI country indices, not on French's site (DGU Appendix A.3)",
    "FF-3-factor": "DGU do not report it: 'very similar to FF-1-factor' (Table 3 note, p. 1931)",
}
DGU_EXCLUDE_LARGEST_SIZE_QUINTILE = True   # DGU footnote 24, p. 1949: the five largest-size portfolios of the 25 are dropped, leaving 20

# ---------------------------------------------------------------------------
# 3. Bands for the replication, written before the code ran
# ---------------------------------------------------------------------------

BAND_SHARPE_ABS = 0.03          # out-of-sample Sharpe ratio within 0.03 of DGU Table 3, per rule and dataset
CHECK_TOP3_RANKING = True        # the ranking of the top three rules within each dataset is preserved
BAND_TURNOVER_REL = 0.25        # turnover within 25% relative of DGU Table 5

# DGU Table 5 reports the unconstrained mean-variance rule's turnover at 606,594
# times that of 1/N on the Industry dataset and 10,466 times on FF-1-factor.
# Those figures are dominated by the months in which the sample covariance is
# near-singular and move by orders of magnitude when French revises a single
# return, so the band applies only where DGU's published relative turnover is
# below the level here; above it, the replicated figure is reported beside the
# published one with no pass or fail. Fixed 10 September 2026.
BAND_TURNOVER_APPLIES_BELOW_RELATIVE = 100.0

# Changed 13 September 2026, after notebook 02 had run, on its evidence: the
# out-of-sample Sharpe band applies only to rules whose published relative
# turnover (DGU Table 5, panel A) is below the same level, and the in-sample
# mean-variance row carries no verdict. Evidence: on the Industry, FF-1 and FF-4
# datasets the unconstrained mean-variance rule moved by 0.12 to 0.25 in Sharpe
# ratio under ten basis points of noise, with the published figure inside the
# range, while the 1/N rows matched to the third decimal. Before the change the
# band applied to every out-of-sample row. The level is the paper's own number,
# already used by the turnover band.
BAND_SHARPE_APPLIES_BELOW_RELATIVE = BAND_TURNOVER_APPLIES_BELOW_RELATIVE
BAND_SHARPE_DECISION_DATE = "2026-09-13"

# Sharpe ratios are compared on the same sample, 1963-07 to 2004-11, 120-month
# window, so the out-of-sample period is 1973-07 to 2004-11.
DGU_EXPECTED_T = 497           # months from 1963-07 to 2004-11 inclusive; a count, checked in notebook 01
DGU_EXPECTED_OOS = 377         # DGU_EXPECTED_T - DGU_ESTIMATION_WINDOW

# The replication extended to the latest month (notebook 07), fixed 1 October
# 2026 before the notebook's code: the nine rules on every month the vintage
# carries, reported for the paper's out-of-sample period, the months since the
# paper and the whole period. The expectations are in the notebook's introduction.
EXTENDED_NEW_PERIOD_START = "2004-12"   # the month after DGU's sample ends

# ---------------------------------------------------------------------------
# 4. DGU's analytical result and simulation (notebook 04)
# ---------------------------------------------------------------------------

# The analytical critical window (DGU Proposition 1, pp. 1937-1938; Figure 1, p. 1940).
# Panel E is the one calibrated to US stock-market data and the one the abstract quotes.
ANALYTIC_SHARPE_TANGENCY = 0.15    # S*, DGU Figure 1 panel E
ANALYTIC_SHARPE_1N = 0.12          # S_ew, DGU Figure 1 panel E
EXPECT_CRITICAL_M_LOWER = {25: 3000, 50: 6000}   # DGU p. 1941: "more than 3000 months" and "more than 6000 months"; the check is order of magnitude and direction in N

# The simulated dataset (DGU section 5.1, pp. 1941-1942).
SIM_RF_MEAN_ANNUAL = 0.02
SIM_RF_SD_ANNUAL = 0.02
SIM_FACTOR_MEAN_ANNUAL = 0.08
SIM_FACTOR_SD_ANNUAL = 0.16
SIM_ALPHA = 0.0
SIM_BETA_RANGE = (0.5, 1.5)          # loadings evenly spread
SIM_IDIO_VOL_RANGE = (0.10, 0.30)    # annual idiosyncratic volatility, uniform, cross-sectional mean 0.20
SIM_N = (10, 25, 50)
SIM_M = (120, 360, 6000)
SIM_T = 24000                        # months in one simulated history
SIM_SEED = 20260910                  # fixed so the run is reproducible; DGU's seed is unknown
# In one 24,000-month history the standard error of a monthly Sharpe ratio is
# about 1/sqrt(24000) = 0.0065, so the comparison with DGU Table 6 uses the same
# band as the empirical tables, plus the sign of (mv minus 1/N) in every cell.
BAND_SIM_SHARPE_ABS = BAND_SHARPE_ABS
# Table 6 reports 1/N and "mv (true)" as full-sample Sharpe ratios and the
# estimated rules out of sample; the comparison follows that convention. As a
# second check every rule is rerun on alternative draws of the market (seeds 1
# to SIM_SPREAD_DRAWS) and the published value is checked against their range.
SIM_SPREAD_DRAWS = 8

# Fat tails (notebook 05), an addition to the paper, fixed 30 September 2026
# before the notebook's code. The market above is rerun with every shock drawn
# from a Student-t distribution, scaled so that the mean and covariance of
# returns are exactly those of the normal market; one scale draw per month is
# shared by all shocks (a multivariate t), so an extreme month hits all assets
# at once. The normal shocks are the same draws as notebook 04's.
SIM_T_DOF = (10, 5)                  # degrees of freedom. 10: excess kurtosis 1, about a broad index's monthly returns. 5: excess kurtosis 6, about an industry portfolio's. At 4 or below the fourth moment is infinite and the standard error of a Sharpe ratio is undefined.
SIM_T_COMMON_SCALE = True            # one scale per month shared by all shocks (multivariate t); False draws one scale per shock series, which changes the shape of the month's co-movement as well as its size
# The crossing table: sample-based mean-variance against 1/N over a grid of
# windows, to locate the window at which the gap closes under each distribution.
SIM_CROSSING_WINDOWS = (120, 240, 360, 600, 1000, 2000, 3000, 4000, 6000, 9000, 12000)
FAT_TAIL_MOMENT_CHECK_ABS = 0.013    # notebook 05, expectation 1: two standard errors of a 24,000-month Sharpe ratio
FAT_TAIL_DIRECTION_SLACK = 0.01      # notebook 05, expectation 2: a cell counts as in the expected direction if Sharpe(t) <= Sharpe(normal) + this
# One test added after notebook 05's first run (6 October 2026), fixed here before
# its code and labelled in the notebook: the fat-tailed market rerun with one
# scale per shock series (common_scale False), so that an extreme month changes
# the shape of the estimated covariance as well as its level. Expectation P1a:
# the mean change in Sharpe ratio across the seven estimated rules and the nine
# cells of the Table 6 grid is more negative than under the common scale at both
# degrees of freedom. P1b: the rules whose weights depend on the shape of the
# covariance and use no means lose more on average than the other estimated rules.
FAT_TAIL_SHAPE_RULES = ("min", "min-c", "g-min-c")   # P1b: the minimum-variance rules

# Volatility that changes through time (notebook 06), the second addition to
# the paper, fixed 1 October 2026 before the notebook's code. The market of
# notebook 04 is rerun with shocks whose volatility follows a GARCH(1,1) process
# (Bollerslev 1986); the factor shock and every asset's idiosyncratic shock get
# their own process with the same parameters, so both the level and the shape
# of the covariance matrix move through time, while the unconditional mean and
# covariance are exactly those of the normal market (omega = 1 - alpha - beta).
# The expectations of notebooks 05 and 06 are in their introductions and use
# the two levels above.
SIM_GARCH_ALPHA = 0.10               # weight of last month's squared shock; a textbook monthly value
SIM_GARCH_BETA = 0.85                # weight of last month's variance; alpha + beta = 0.95 is the persistence, a half-life of a volatility shock of about 13 months
SIM_GARCH_OWN_PROCESS_PER_ASSET = True   # the factor and each idiosyncratic shock follow their own process; False would scale all shocks by one common volatility, which leaves the shape of the covariance unchanged

# ---------------------------------------------------------------------------
# 5. The extension: constrained index tracking on the 49 industries
# ---------------------------------------------------------------------------

EXT_UNIVERSE = "ind49"
EXT_ESTIMATION_WINDOW = 120        # the same window as the replication, so one convention runs through the repository
EXT_REBALANCE = "monthly"
# The extension sample starts at the first month in which all 49 industries carry
# a return (French codes the early gaps as -99.99): Hlth is the last industry to
# fill and does so in 1969-07. Notebook 01 asserts it on every run.
EXT_SAMPLE_START = "1969-07"
EXT_SAMPLE_END = None              # the latest month the vintage carries

# The benchmark (the index the portfolio tracks): the cap-weighted combination
# of the 49 industries, weights from firm count times average firm size, must
# track the market factor plus the risk-free rate within this tracking error a
# year over the full sample (notebook 08). If it does not, the derived weights
# are dropped and the benchmark is the market series.
CAPW_TE_MAX_ANNUAL = 0.0050

# Notebook 08, the cap-weight check, fixed 1 October 2026 before its code. The
# weight of industry i in month t is its firm count times its average firm size
# in French's file for month t, divided by the sum over the 49 industries;
# French measures both at the start of month t, so the weight uses nothing
# unknown when the month begins. The previous-month reading is run beside it as
# a check of this timing, and the notebook stops if it tracks more closely.
CAPW_WEIGHT_TIMING = "same_month"
CAPW_WEIGHT_TIMING_ALTERNATIVE = "previous_month"   # reported beside the chosen timing, never used as the benchmark
CAPW_MARKET_SERIES = ("Mkt-RF", "RF")                # the market series is Mkt-RF plus RF, so that both sides are plain returns
CAPW_REPORT_MONTHS = ("first", "2004-11", "last")

# The direct test of where the benchmark's gap to the market comes from
# (notebook 08, second part), fixed 1 October 2026 before its code. French forms
# the industry portfolios once a year at the end of June from the firms that
# have an SIC code then, while the market return takes in a firm from its first
# full month of trading, so between one July and the next the market holds
# firms that no industry portfolio holds. The test needs the firm list of each
# portfolio each month, which French does not publish, so it runs on the CRSP
# monthly stock file (WRDS, under Queen Mary University of London's
# subscription), the one use of data outside French's library in the study.
# The CRSP data stay on the machine that holds them: no notebook reads them,
# the repository carries none, and only the aggregates in
# DIRECT_TEST_RESULTS_FILE are published, with CRSP cited by name. The market
# universe is CRSP share codes 10 and 11 on NYSE, AMEX and NASDAQ, weighted by
# start-of-month market equity; the June-formed universe is the firms of that
# market that were in it at the end of June of the formation year with a
# positive SIC code. The script is checks/crsp_direct_test.py; its pure
# functions are in bp.direct_test.
DIRECT_TEST_SHARE_CODES = (10, 11)
DIRECT_TEST_EXCHANGE_CODES = (1, 2, 3)
DIRECT_TEST_FORMATION_MONTH = 6          # the end of June
DIRECT_TEST_MARKET_TE_MAX = 0.0020       # expectation 1: the CRSP market universe tracks French's market return within 20 basis points a year
DIRECT_TEST_MIN_CORRELATION = 0.80       # expectation 2: correlation of the predicted gap with the observed one
DIRECT_TEST_SLOPE_RANGE = (0.8, 1.2)     # expectation 2: slope of the observed gap on the predicted one
DIRECT_TEST_RESIDUAL_TE_MAX = 0.0025     # expectation 4: tracking error of what the explanation leaves, 25 basis points a year
DIRECT_TEST_RESULTS_FILE = "checks/results_crsp_direct_test.json"   # aggregates only; committed

# Constraints. Fixed 10 September 2026; the sources for the levels are in
# notebook 11, "Where the numbers come from".
ACTIVE_WEIGHT_BOUND = 0.02         # each untilted industry within 2 percentage points of its benchmark weight; an absolute cap does not fit 49 industries whose benchmark weights run from under 0.5% to over 10%
TURNOVER_CAP_MONTHLY_ONE_WAY = 0.02   # one-way turnover per rebalance, against drifted weights; 2% a month is about 24% a year, the enhanced-indexing range
TILT_INDUSTRIES = ("Coal", "Oil", "Util")   # French's column names, stripped of padding
TILT_MAX_SHARE_OF_BENCHMARK = 0.5  # each tilted industry held at no more than half its benchmark weight, a 50% reduction relative to the benchmark; a tilted industry's bounds are [0, 0.5 * benchmark], and ACTIVE_WEIGHT_BOUND does not apply to it, because the two would conflict for any industry above 4% of the benchmark
LONG_ONLY = True

# Constraint sets, cumulative, reported in this order. C3 is the mandate: the
# tilt with beta neutrality and the exclusions. Beta neutrality and the
# exclusions were added on 4 October 2026, after notebook 11's checks D1 to D4
# and before notebook 12's code, because the tilt's hedge left a beta bet
# standing when the covariance was noisy and bought tobacco and weapons. The
# tilt alone is CONSTRAINT_SET_TILT_ONLY, the C3 of the first design, reported
# beside the mandate in notebooks 11 and 12 so that the cost of the two
# additions is visible.
CONSTRAINT_SETS = {
    "C0": ("long_only",),
    "C1": ("long_only", "active_weight_bound"),
    "C2": ("long_only", "active_weight_bound", "turnover_cap"),
    "C3": ("long_only", "active_weight_bound", "turnover_cap", "tilt", "beta_neutral", "exclusion"),
}
CONSTRAINT_SET_TILT_ONLY = ("long_only", "active_weight_bound", "turnover_cap", "tilt")   # the C3 of the design of 10 September 2026
MANDATE_EXCLUSIONS = ("Smoke", "Guns")   # tobacco and weapons, the standard exclusions of Dutch institutional mandates, in French's industry names; weight zero

# The data the covariance estimators see, fixed 1 October 2026. The 120-month
# window is the paper's convention and gives 49 x 120 = 5,880 returns for the
# 1,225 entries of a 49-industry covariance matrix, fewer than five per entry.
# Practitioners estimate from daily returns (three years hold about 756 trading
# days, 30 observations per entry), impose structure, and weight recent days
# more; the extension does all three. Each estimator is built at two data
# frequencies: "monthly_120", the last EXT_ESTIMATION_WINDOW months of monthly
# excess returns, and "daily_3y", the last EXT_COV_DAILY_WINDOW_YEARS years of
# daily excess returns scaled to a monthly covariance by DAILY_TO_MONTHLY_SCALE
# (notebook 10 tests that scaling). A trading day on which any industry has no
# return (one in the sample, Softw on 1971-03-11) is left out of the window.
EXT_COV_FREQUENCIES = ("monthly_120", "daily_3y")
EXT_COV_DAILY_WINDOW_YEARS = 3
EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY = (1, 5)   # reported for the sample estimator only, no verdict
DAILY_TO_MONTHLY_SCALE = 21                        # 252 trading days a year divided by 12
EXT_COV_EWMA_HALFLIFE_DAYS = 252                   # the recency-weighted sensitivity: a half-life of one year of trading days, the RiskMetrics setting of Dom, Howard, Jansen and Lohre (2024, section 3.3.2); 126 until 3 October 2026, changed on reading that paper, before notebook 10's code

# Covariance estimators. The shrinkage target follows Dom, Howard, Jansen and
# Lohre (2024), the paper the design named as its reference, checked on
# 3 October 2026 before notebook 10's code: their linear shrinkage pulls the
# sample covariance towards a scaled identity matrix (Ledoit and Wolf 2004b),
# and their footnote 11 reports that the constant-correlation target gave
# inferior results. The same paper estimates from three years of daily returns
# and rebalances monthly, and finds that under long-only constraints the
# differences between estimators shrink, which is notebook 10's expectation 3.
COV_ESTIMATORS = ("sample", "ledoit_wolf", "factor", "pca")
LW_TARGET = "scaled_identity"          # Ledoit and Wolf (2004b), "A well-conditioned estimator for large-dimensional covariance matrices"; was "constant_correlation" until 3 October 2026
LW_TARGET_SENSITIVITY = "constant_correlation"   # Ledoit and Wolf (2004a), "Honey, I Shrunk the Sample Covariance Matrix"; reported, no verdict
FACTOR_MODEL_FACTORS = ("Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom")   # the Fama-French five factors plus momentum; the estimator is B F B' plus a diagonal of residual variances, B estimated in the same estimation window
PCA_N_COMPONENTS = 5                   # five components plus a diagonal idiosyncratic term
PCA_PC1_MARKET_CORR_MIN = 0.95         # notebook 09, expectation 2: the first component's correlation with the cap-weighted market return
PCA_USE_COVARIANCE = True              # covariance, not correlation, of monthly excess returns
PCA_SIGN_RULE = "positive market loading"
PCA_WINDOWS = {                        # three pre-specified windows; loading stability is the finding and carries no pass or fail
    "to_2019":   (None, "2019-12"),
    "2020_2022": ("2020-01", "2022-12"),
    "from_2023": ("2023-01", None),
}

# Notebook 09, the principal components of the 49 industries, fixed 1 October
# 2026 before its code; the expectations are in its introduction.
PCA_REPORT_COMPONENTS = 10
PCA_PARALLEL_DRAWS = 200
PCA_PARALLEL_PERCENTILE = 95
PCA_PARALLEL_SEED = 20260910
# Set after notebook 09 ran, as its design said: the first component's
# correlation with the benchmark fell below PCA_PC1_MARKET_CORR_MIN in the
# window from 2023 (0.837), so notebook 10's PCA estimator takes the
# benchmark's excess return as its first factor. Parallel analysis found two
# components above noise; PCA_N_COMPONENTS stays at five and the two-component
# version is reported beside it.
PCA_FIRST_COMPONENT = "market"        # notebook 10: the benchmark's excess return is the first factor; "data" would take it from the eigenvectors
PCA_N_COMPONENTS_SENSITIVITY = 2      # the parallel-analysis count on the full sample, reported beside PCA_N_COMPONENTS

# Notebook 10, the four covariance estimators, fixed 3 October 2026 before its
# code. The estimators, data frequencies, tests and expectations are stated in
# the notebook's introduction and in covariance.py.
COV_EVAL_START = "1979-07"            # 1969-07 plus 120 months; the three-year daily window starts 1972-07 and is the shorter requirement
COV_TEST_ACTIVE_DRAWS = 200           # random active-weight vectors for the calibration test
COV_TEST_SEED = 20260910
COV_LONG_ONLY_RANGE_MAX = 0.01        # expectation 3: one percentage point of annualised volatility across the four estimators on daily data
COV_SCALE_RATIO_RANGE = (0.8, 1.25)   # expectation 4: the ratio of the benchmark's realised monthly variance to 21 times its daily variance
# Three tests added after notebook 10's first run (4 October 2026), each fixed
# here before its code and labelled in the notebook: P1, a scale correction by
# the variance ratio of Lo and MacKinlay (1988), one number from the benchmark's
# daily returns that multiplies the daily covariance in place of
# DAILY_TO_MONTHLY_SCALE alone; P2, bias statistics by decade; P3, trading costs
# of the minimum-variance portfolios at EXT_COST_LEVELS.
COV_SCALE_VR_HORIZON_DAYS = DAILY_TO_MONTHLY_SCALE   # the horizon of the variance ratio: one month of trading days
COV_SCALE_VR_WINDOW_YEARS_SENSITIVITY = 10           # P1 sensitivity: the variance ratio from ten years of daily benchmark returns
COV_POST_RUN_DECADES_MIN_CLOSER = 4                  # P1a: of five decades, the corrected forecast is closer to calibrated in at least this many
COV_POST_RUN_COST_MIN_VARIANTS = 6                   # P3a and P3b: of eight main variants, the long-only portfolio beats the unconstrained one net of cost in at least this many

# Notebook 11, the optimiser, fixed 4 October 2026 before its code; the
# constraints themselves date from 10 September. The problem, the infeasible-month
# rule and the expectations are stated in the notebook's introduction and in
# optimiser.py.
OPT_SOLVER = "CLARABEL"                  # interior-point solver shipped with cvxpy; deterministic
OPT_SOLVER_OPTIONS = {"tol_gap_abs": 1e-12, "tol_gap_rel": 1e-12, "tol_feas": 1e-10}   # tighter than the solver's defaults (1e-8), which left weights 1e-4 off an independent solver's on a test problem
OPT_SOLVER_OPTIONS_ROBUST = {"tol_gap_abs": 1e-9, "tol_gap_rel": 1e-9, "tol_feas": 1e-9}   # the robust problem is written with second-order cones, on which the solver reaches 1e-9 and reports 1e-12 as inaccurate; the objective agrees with the 1e-12 run to 1e-10 on the test problems
OPT_TOL = 1e-6                           # feasibility checks on weights, bounds and turnover; also "binding" means within this of the limit
OPT_TOL_TE = 1e-6                        # expectation 2a: a forecast tracking error below this, per year, counts as zero
OPT_TILT_TE_MAX_ANNUAL = 0.01            # expectation 3: 100 basis points a year, the enhanced-indexing ex-ante limit (Robeco's composite ran at 1.06%, J.P. Morgan quotes below 2%; sources in notebook 11)
OPT_REPORT_MONTHS = ("first_eval", "2004-11", "last")   # first_eval is COV_EVAL_START, the first month with both windows
OPT_SPEED_LIMIT_STEPS = (1, 6, 12, 24)   # expectation 5: rebalances reported on the journey from 1/N to the benchmark
# Checks D1 to D4 on the practical problem behind the tilt's hedge, added after
# notebook 11's first run (4 October 2026), each fixed here before its code.
OPT_D1_MARKET_SHARE_MAX = 0.05         # D1a: the market part of the tilt's forecast active variance is below this share in every month-estimator pair
OPT_D2_TE_RISE_MAX = 0.0001            # D2a: a beta-neutral constraint raises the forecast tracking error by less than one basis point a year in every pair
OPT_D3_EXCLUDE = ("Smoke",)            # D3: the hedge's tobacco purchase refused, its cost reported
# The fixes F1 to F3 (4 October 2026, before notebook 12's code): beta neutrality
# and the exclusions join C3 (above), and a robust portfolio, the weights whose
# largest forecast active variance across the estimators' covariances is
# smallest (Goldfarb and Iyengar 2003; Tutuncu and Koenig 2004), joins the
# rolling evaluation as a variant. Its beta neutrality holds under the average
# of the five beta vectors, because one equality per covariance was infeasible
# at 1979-07 under the mandate's bounds.
OPT_FIX_RISE_MAX = 0.0015              # F2: the mandate's C3 costs less than 15 basis points a year more than the tilt alone
ROBUST_VARIANT = ("robust", "all_five")   # the robust portfolio: minimax over the five covariances
# Check D5 on the solver, added after notebook 12's first run (6 October 2026),
# fixed here before its code: at this month the index moved more than the cap
# allows, no portfolio satisfied the cap and the tilt at once, and the solver
# reported "optimal" with weights that fail the independent check. The
# optimiser therefore finds the smallest feasible turnover first and never
# poses a capped problem that has no solution; the check of every returned
# portfolio stays.
OPT_D5_MONTH = "1984-07"

# Notebook 12, the rolling evaluation, fixed 4 October 2026 before its code: one
# path per specification, each portfolio built on the data before its month,
# judged on realised tracking error, calibration of the forecasts, active
# return net of costs, turnover and holdings. The expectations are in the
# notebook's introduction.
EVAL_TE_RANGE_MAX = 0.0010           # expectation 2: 10 basis points a year across the four daily estimators
EVAL_BIAS_RANGE = (0.8, 1.25)        # expectation 3, the same band as COV_SCALE_RATIO_RANGE
EVAL_RELAX_SHARE_MAX = 0.10          # expectation 6: at most one month in ten with a relaxed cap
EVAL_START_PORTFOLIO = "benchmark"   # every path starts as the index
EVAL_BENCHMARK_TOL = 1e-6            # expectation 1: an active return below this a month counts as holding the benchmark; written as 1e-10 and set to the solver's precision (about 1e-7) before the recorded run
# Expectations 8 and 9, added after notebook 12's first run (6 October 2026),
# each fixed here before its code and labelled in the notebook. RiskMetrics
# joins as a seventh variant because Dom, Howard, Jansen and Lohre (2024) find
# that time dynamics matter more than shrinkage or factor structure; the
# calibration multiplies each forecast by the ratio of realised to forecast
# tracking error over the previous EVAL_CALIBRATION_WINDOW months of the same
# path, using only months before the forecast.
EWMA_VARIANT = ("sample", "daily_5y_ewma")   # the seventh variant: RiskMetrics, as in notebook 10's sensitivity
EVAL_CALIBRATION_WINDOW = 60                 # months of a path's own past the calibration uses; a standard deviation from 60 observations has a standard error of about 9%
EVAL_CALIBRATION_MIN = 24                    # the forecast is left as it is while fewer months are available
EVAL_CALIBRATED_BIAS_RANGE = (0.85, 1.15)    # expectation 9: the calibrated bias statistic under C3, four daily estimators

# Attribution (notebook 13): factor-based attribution of each constrained
# portfolio's realised active return against the factors above, contribution
# by factor plus residual, with the identity as the check.
ATTRIBUTION_FACTORS = FACTOR_MODEL_FACTORS
ATTRIBUTION_IDENTITY_MAX_ABS_ERROR = 1e-9        # contributions plus residual minus active return, per month, in return units

# Specification count, written down before the code runs and reported.
EXT_SPEC_COUNT = (len(COV_ESTIMATORS) + 3) * len(CONSTRAINT_SETS)   # 28 optimised specifications: four daily-based estimators, the monthly sample covariance, the robust portfolio and RiskMetrics, times four constraint sets
EXT_COST_LEVELS = (DGU_TCOST, TCOST_SENSITIVITY)              # each reported net of cost at both levels; the cost is applied after the fact and does not enter the optimisation

# ---------------------------------------------------------------------------
# 6. Output conventions
# ---------------------------------------------------------------------------

OUTPUT_DIR = "outputs"
PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

In [ ]:
%%writefile src/bp/french_loader.py
"""
Ken French data library: download, record, parse.

The library serves each dataset as a zip holding one CSV. The CSV is not a
single table. It is a text file with between two and ten blocks, each announced by a title
line ("  Average Value Weighted Returns -- Monthly"), followed by a header row
that starts with a comma, followed by rows that start with a date: six digits
for monthly data (192607), four for annual (1927). Missing values are coded
-99.99 or -999. Returns are in percent.

This module turns that into pandas DataFrames indexed by a monthly Period, in
decimal returns, with the missing codes replaced by NaN. It also writes a
provenance record for every file it downloads: the URL, the download time, the
SHA-256 of the zip, and the CRSP vintage line French prints at the top of each
CSV. A result is only as reproducible as the vintage it was computed on, and
French revises history, so the record is part of every output.

Nothing here is committed to the repository: the zips live in a local cache
directory that .gitignore excludes.
"""
from __future__ import annotations

import hashlib
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests

from . import constants as C

_DATE_ROW = re.compile(r"^\s*(\d{8}|\d{6}|\d{4})\s*,")   # daily YYYYMMDD, monthly YYYYMM, annual YYYY


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def download(key: str, cache_dir: str | Path = "data/french", force: bool = False) -> dict:
    """Fetch one French zip by its key in constants.FRENCH_FILES.

    Returns the provenance record for the file and writes the zip to cache_dir.
    A cached zip is reused unless force=True, and its record is rebuilt from the
    cached bytes, so the SHA-256 always describes the file actually on disk.
    """
    filename = C.FRENCH_FILES[key]
    url = C.FRENCH_BASE_URL + filename
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / filename
    if force or not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        fetched_now = True
    else:
        fetched_now = False
    data = path.read_bytes()
    now_utc = datetime.now(timezone.utc)
    record = {
        "key": key,
        "filename": filename,
        "url": url,
        "fetched_this_run": fetched_now,
        "download_time_utc": now_utc.isoformat(timespec="seconds"),
        "download_time_local": now_utc.astimezone(ZoneInfo(C.TIMEZONE)).isoformat(timespec="seconds"),
        "zip_sha256": sha256_bytes(data),
        "zip_bytes": len(data),
        "vintage_line": _vintage_line(data),
    }
    return record


def _csv_text(zip_bytes: bytes) -> str:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if len(names) != 1:
            raise ValueError(f"expected one CSV in the zip, found {names}")
        return zf.read(names[0]).decode("latin-1")


def _vintage_line(zip_bytes: bytes) -> str:
    """The first line of the CSV, e.g. 'This file was created using the 202607 CRSP database.'"""
    text = _csv_text(zip_bytes)
    for line in text.splitlines():
        if line.strip():
            first = line.strip()
            return first.split(".")[0] + "." if "." in first else first
    return ""


def parse_blocks(text: str) -> list[dict]:
    """Split a French CSV into its blocks.

    Each block is returned as {"title": str, "frequency": "daily" | "monthly" |
    "annual", "frame": DataFrame} where the frame is indexed by Period (day,
    month or year) and holds floats in the file's own units (percent). The title is the last non-empty line before
    the header row that is not itself a data row.
    """
    lines = text.splitlines()
    blocks = []
    i = 0
    n = len(lines)
    while i < n:
        if _DATE_ROW.match(lines[i]):
            # Walk back to the header row (starts with a comma) and the title above it.
            header_idx = i - 1
            while header_idx >= 0 and not lines[header_idx].lstrip().startswith(","):
                header_idx -= 1
            if header_idx < 0:
                raise ValueError(f"data row at line {i} has no header row above it")
            title_idx = header_idx - 1
            while title_idx >= 0 and not lines[title_idx].strip():
                title_idx -= 1
            title = lines[title_idx].strip() if title_idx >= 0 else ""
            # The factor files and the later blocks of the size-and-value file
            # carry no title line, only prose; a sentence is not a title.
            if not title or title.endswith(".") or len(title) > 70:
                title = f"(untitled block {len(blocks)})"
            columns = [c.strip() for c in lines[header_idx].split(",")[1:]]
            rows = []
            j = i
            while j < n and _DATE_ROW.match(lines[j]):
                rows.append(lines[j])
                j += 1
            raw = pd.read_csv(io.StringIO("\n".join(rows)), header=None, index_col=0)
            raw.columns = columns[: raw.shape[1]]
            date_len = len(str(raw.index[0]).strip())
            if date_len == 8:
                idx = pd.PeriodIndex(pd.to_datetime(raw.index.astype(str), format="%Y%m%d"), freq="D")
                freq = "daily"
            elif date_len == 6:
                idx = pd.PeriodIndex([pd.Period(f"{str(d)[:4]}-{str(d)[4:6]}", freq="M") for d in raw.index])
                freq = "monthly"
            else:
                idx = pd.PeriodIndex([pd.Period(str(d), freq="Y") for d in raw.index])
                freq = "annual"
            frame = raw.astype(float)
            frame.index = idx
            frame.index.name = "period"
            blocks.append({"title": title, "frequency": freq, "frame": frame})
            i = j
        else:
            i += 1
    return blocks


def load_blocks(key: str, cache_dir: str | Path = "data/french") -> list[dict]:
    """All blocks of one file, with the file's provenance record attached to each."""
    record = download(key, cache_dir)
    text = _csv_text((Path(cache_dir) / C.FRENCH_FILES[key]).read_bytes())
    blocks = parse_blocks(text)
    for b in blocks:
        b["provenance"] = record
    return blocks


def load_monthly(key: str, block: int | str = 0, cache_dir: str | Path = "data/french",
                 as_decimal: bool = True) -> pd.DataFrame:
    """One block of one file, in decimal returns with missing codes as NaN.

    The name dates from the monthly files; the function serves the daily files
    in the same way (the block's index is then a daily Period).

    block is an integer position among the file's blocks, or a regular
    expression matched against block titles (the first match wins).
    """
    blocks = load_blocks(key, cache_dir)
    if isinstance(block, int):
        chosen = blocks[block]
    else:
        matches = [b for b in blocks if re.search(block, b["title"], flags=re.I)]
        if not matches:
            raise KeyError(f"no block in {key} matches {block!r}; titles: {[b['title'] for b in blocks]}")
        chosen = matches[0]
    frame = chosen["frame"].copy()
    for code in C.FRENCH_MISSING_CODES:
        frame = frame.mask(np.isclose(frame, code))
    if as_decimal and C.FRENCH_RETURNS_ARE_PERCENT:
        frame = frame / 100.0
    frame.attrs["title"] = chosen["title"]
    frame.attrs["provenance"] = chosen["provenance"]
    return frame


def load_counts(key: str, block: int | str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """A block in the file's own units (firm counts, average firm size), missing codes as NaN."""
    return load_monthly(key, block, cache_dir, as_decimal=False)


def write_provenance(records: list[dict], path: str | Path = C.PROVENANCE_FILE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ordered = sorted(records, key=lambda r: r["key"])
    path.write_text(json.dumps(ordered, indent=2, sort_keys=True) + "\n")
    return path


def block_summary(key: str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """One row per block: title, frequency, columns, rows, first and last period."""
    rows = []
    for k, b in enumerate(load_blocks(key, cache_dir)):
        f = b["frame"]
        rows.append({
            "file": key, "block": k, "title": b["title"], "frequency": b["frequency"],
            "columns": f.shape[1], "rows": f.shape[0],
            "first": str(f.index[0]), "last": str(f.index[-1]),
        })
    return pd.DataFrame(rows)

In [ ]:
%%writefile src/bp/dgu_published.py
"""
The published numbers of DeMiguel, Garlappi and Uppal (2009), Review of
Financial Studies 22(5), transcribed from Tables 3, 4, 5 and 6, for the four
datasets built from Ken French's library.

The comparison tables in the notebooks are built from this module, never
typed into a notebook. tests/test_dgu_published.py re-reads the numbers from
the journal PDF when it is present on the machine (papers/, not committed) and
fails on any digit that differs.

Monthly figures throughout. Sharpe ratios and CEQ returns are for the
out-of-sample period 1973-07 to 2004-11 with a 120-month window (Tables 3 and
4); "mv (in sample)" uses the full sample. Turnover (Table 5) is the 1/N
rule's absolute monthly turnover and every other rule's turnover
relative to it; return-loss is the extra monthly return a rule needs to
match 1/N's Sharpe ratio net of a 50 basis point proportional cost.
"""

DATASETS = ("Industry", "MKT/SMB/HML", "FF-1-factor", "FF-4-factor")

# Table 3, p. 1931: monthly Sharpe ratio, with the Jobson-Korkie p-value of the
# difference from 1/N in the second tuple (None where the paper prints none).
SHARPE = {
    "ew":              ((0.1353, 0.2240, 0.1623, 0.1753), None),
    "mv (in sample)":  ((0.2124, 0.2851, 0.5098, 0.5364), None),
    "mv":              ((0.0679, 0.2186, 0.0128, 0.1841), (0.17, 0.46, 0.02, 0.45)),
    "bs":              ((0.0719, 0.2536, 0.0138, 0.1791), (0.19, 0.25, 0.02, 0.48)),
    "dm":              ((0.0581, 0.0016, 0.0004, 0.2355), (0.14, 0.00, 0.01, 0.17)),
    "min":             ((0.1554, 0.2493, 0.2778, -0.0183), (0.30, 0.23, 0.01, 0.01)),
    "vw":              ((0.1138, 0.1138, 0.1138, 0.1138), (0.01, 0.00, 0.01, 0.00)),
    "mp":              ((0.0533, -0.0002, 0.1238, 0.1230), (0.04, 0.00, 0.08, 0.03)),
    "mv-c":            ((0.0678, 0.1084, 0.1977, 0.2024), (0.03, 0.02, 0.02, 0.27)),
    "bs-c":            ((0.0819, 0.1514, 0.1955, 0.2062), (0.06, 0.09, 0.03, 0.25)),
    "min-c":           ((0.1425, 0.2493, 0.1546, 0.3580), (0.41, 0.23, 0.35, 0.00)),
    "g-min-c":         ((0.1451, 0.2467, 0.1615, 0.3028), (0.31, 0.25, 0.47, 0.00)),
    "mv-min":          ((0.0772, 0.2546, -0.0079, 0.1757), (0.21, 0.22, 0.01, 0.50)),
    "ew-min":          ((0.1576, 0.2503, 0.2608, -0.0161), (0.21, 0.17, 0.00, 0.01)),
}

# Table 4, p. 1934: monthly certainty-equivalent return, gamma = 1, with p-values.
CEQ = {
    "ew":              ((0.0050, 0.0039, 0.0073, 0.0072), None),
    "mv (in sample)":  ((0.0106, 0.0047, 0.0300, 0.0304), None),
    "mv":              ((-0.7816, 0.0045, -2.7142, -0.0829), (0.00, 0.31, 0.00, 0.01)),
    "bs":              ((-0.3157, 0.0043, -0.6504, -0.0362), (0.00, 0.32, 0.00, 0.06)),
    "dm":              ((-0.0319, -0.0084, -0.0296, 0.0110), (0.01, 0.04, 0.00, 0.11)),
    "min":             ((0.0052, 0.0039, 0.0100, -0.0002), (0.45, 0.45, 0.12, 0.00)),
    "vw":              ((0.0042, 0.0042, 0.0042, 0.0042), (0.04, 0.44, 0.00, 0.00)),
    "mp":              ((0.0014, -0.0026, 0.0054, 0.0053), (0.05, 0.04, 0.09, 0.10)),
    "mv-c":            ((0.0023, 0.0030, 0.0090, 0.0075), (0.10, 0.28, 0.03, 0.42)),
    "bs-c":            ((0.0031, 0.0038, 0.0088, 0.0074), (0.15, 0.46, 0.05, 0.44)),
    "min-c":           ((0.0047, 0.0039, 0.0060, 0.0051), (0.40, 0.45, 0.12, 0.17)),
    "g-min-c":         ((0.0048, 0.0038, 0.0067, 0.0070), (0.41, 0.40, 0.17, 0.45)),
    "mv-min":          ((-0.2337, 0.0044, -0.0875, -0.0318), (0.00, 0.28, 0.00, 0.07)),
    "ew-min":          ((0.0052, 0.0039, 0.0093, -0.0002), (0.42, 0.43, 0.12, 0.00)),
}

# Table 5, p. 1935. TURNOVER_1N is absolute; TURNOVER_RELATIVE is each rule's
# turnover divided by 1/N's; RETURN_LOSS is panel B.
TURNOVER_1N = (0.0216, 0.0237, 0.0162, 0.0198)
TURNOVER_RELATIVE = {
    "mv":       (606594.36, 2.83, 10466.10, 3553.03),
    "bs":       (10621.23, 1.85, 11796.47, 3417.81),
    "dm":       (21744.35, 76.30, 918.40, 32.46),
    "min":      (21.65, 1.11, 45.47, 6.83),
    "vw":       (0.0, 0.0, 0.0, 0.0),
    "mp":       (11.98, 59.41, 2.39, 2.07),
    "mv-c":     (7.17, 4.12, 17.53, 13.82),
    "bs-c":     (7.22, 3.65, 17.32, 13.07),
    "min-c":    (2.58, 1.11, 3.93, 1.76),
    "g-min-c":  (1.52, 1.09, 1.78, 1.70),
    "mv-min":   (9927.09, 2.61, 4292.16, 4857.19),
    "ew-min":   (15.66, 1.11, 34.10, 6.80),
}
RETURN_LOSS = {
    "mv":       (231.8504, 0.0003, 7.4030, 1.5740),
    "bs":       (9.4602, -0.0004, 2.0858, 1.1876),
    "dm":       (8.9987, 0.0393, 0.1302, -0.0007),
    "min":      (0.0015, -0.0004, -0.0008, 0.0024),
    "vw":       (0.0037, 0.0157, 0.0021, 0.0028),
    "mp":       (0.0050, 0.0227, 0.0023, 0.0030),
    "mv-c":     (0.0048, 0.0041, -0.0005, 0.0002),
    "bs-c":     (0.0038, 0.0023, -0.0004, -0.0000),
    "min-c":    (-0.0001, -0.0004, 0.0006, -0.0025),
    "g-min-c":  (-0.0003, -0.0003, 0.0001, -0.0029),
    "mv-min":   (6.8115, -0.0003, 0.9306, 1.8979),
    "ew-min":   (0.0008, -0.0004, -0.0011, 0.0024),
}

# Table 6, p. 1943: simulated data, monthly Sharpe ratio, columns ordered
# (N=10: M=120, 360, 6000), (N=25: same), (N=50: same).
SIM_COLUMNS = tuple((n, m) for n in (10, 25, 50) for m in (120, 360, 6000))
SIM_SHARPE = {
    "ew":        (0.1356, 0.1356, 0.1356, 0.1447, 0.1447, 0.1447, 0.1466, 0.1466, 0.1466),
    "mv (true)": (0.1477, 0.1477, 0.1477, 0.1477, 0.1477, 0.1477, 0.1477, 0.1477, 0.1477),
    "mv":        (-0.0019, 0.0077, 0.1416, 0.0027, 0.0059, 0.1353, 0.0078, -0.0030, 0.1212),
    "bs":        (-0.0021, 0.0087, 0.1416, 0.0031, 0.0074, 0.1363, 0.0076, -0.0035, 0.1229),
    "dm":        (0.0725, 0.1475, 0.1464, 0.0133, 0.1473, 0.1457, 0.0201, 0.0380, 0.1430),
    "min":       (0.1113, 0.1181, 0.1208, 0.0804, 0.0911, 0.0956, 0.0491, 0.0676, 0.0696),
    "mp":        (0.1171, 0.1349, 0.1354, 0.1265, 0.1442, 0.1446, 0.1312, 0.1460, 0.1465),
    "mv-c":      (0.0970, 0.1121, 0.1276, 0.1011, 0.1150, 0.1315, 0.1111, 0.1194, 0.1355),
    "bs-c":      (0.1039, 0.1221, 0.1317, 0.1095, 0.1222, 0.1350, 0.1162, 0.1251, 0.1381),
    "min-c":     (0.1284, 0.1324, 0.1335, 0.1181, 0.1227, 0.1248, 0.1224, 0.1277, 0.1292),
    "g-min-c":   (0.1289, 0.1312, 0.1320, 0.1311, 0.1336, 0.1348, 0.1364, 0.1402, 0.1415),
    "mv-min":    (-0.0029, 0.0106, 0.1414, 0.0087, 0.0172, 0.1361, 0.0016, -0.0068, 0.1229),
    "ew-min":    (0.1116, 0.1184, 0.1211, 0.0810, 0.0918, 0.0964, 0.0496, 0.0684, 0.0706),
}


def table(name: str):
    """A pandas DataFrame of one published table, rules by dataset."""
    import pandas as pd
    if name == "sharpe":
        return pd.DataFrame({k: v[0] for k, v in SHARPE.items()}, index=DATASETS).T
    if name == "ceq":
        return pd.DataFrame({k: v[0] for k, v in CEQ.items()}, index=DATASETS).T
    if name == "turnover_relative":
        return pd.DataFrame(TURNOVER_RELATIVE, index=DATASETS).T
    if name == "return_loss":
        return pd.DataFrame(RETURN_LOSS, index=DATASETS).T
    if name == "sim_sharpe":
        cols = pd.MultiIndex.from_tuples(SIM_COLUMNS, names=["N", "M"])
        return pd.DataFrame(SIM_SHARPE, index=cols).T
    raise KeyError(name)

In [ ]:
%%writefile src/bp/datasets.py
"""
The four DGU datasets built from Ken French's library.

DGU (Table 2, p. 1918; Appendix A, pp. 1948-1949) work in monthly excess
returns over the Treasury bill rate from French's site. Each dataset is a
set of risky assets plus one or more factor portfolios:

  Industry      ten industry portfolios plus the market factor, N = 11
  MKT/SMB/HML   the three Fama-French factors as the assets, N = 3
  FF-1-factor   twenty size-and-value portfolios plus the market, N = 21
  FF-4-factor   the same twenty plus MKT, SMB, HML and UMD, N = 24

The twenty are the 25 size-and-value portfolios less the five with the
largest firms (DGU footnote 24). SMB, HML and UMD are zero-cost portfolios,
so their return is already an excess return; the industry and size-and-value
portfolios are long-only, so RF is subtracted from them.
"""
from __future__ import annotations

import pandas as pd

from . import constants as C
from . import french_loader as fl


def _window(frame: pd.DataFrame, start: str | None, end: str | None) -> pd.DataFrame:
    lo = pd.Period(start, "M") if start else frame.index[0]
    hi = pd.Period(end, "M") if end else frame.index[-1]
    return frame.loc[lo:hi]


def load_inputs(cache_dir: str = "data/french") -> dict[str, pd.DataFrame]:
    """Every French block the datasets need, in decimal returns."""
    return {
        "factors3": fl.load_monthly("factors3", 0, cache_dir),
        "momentum": fl.load_monthly("momentum", 0, cache_dir),
        "ind10": fl.load_monthly("ind10", 0, cache_dir),
        "size_bm25": fl.load_monthly("size_bm25", 0, cache_dir),
    }


def twenty_of_25(p25: pd.DataFrame) -> pd.DataFrame:
    big = [c for c in p25.columns if c.startswith("BIG") or c.startswith("ME5")]
    if len(big) != 5:
        raise ValueError(f"expected five largest-size portfolios, found {big}")
    return p25.drop(columns=big)


def build(name: str, inputs: dict[str, pd.DataFrame] | None = None,
          start: str | None = C.DGU_SAMPLE_START, end: str | None = C.DGU_SAMPLE_END,
          cache_dir: str = "data/french") -> pd.DataFrame:
    """One DGU dataset as a DataFrame of monthly excess returns, assets in columns.

    The factor columns come last, in the order DGU list them, so that the
    market factor is the last column of the Industry and FF-1-factor sets.
    """
    if inputs is None:
        inputs = load_inputs(cache_dir)
    f3, mom, i10, p25 = inputs["factors3"], inputs["momentum"], inputs["ind10"], inputs["size_bm25"]
    rf = f3["RF"]
    if name == "Industry":
        risky = i10.sub(rf, axis=0)
        out = pd.concat([risky, f3[["Mkt-RF"]]], axis=1)
    elif name == "MKT/SMB/HML":
        out = f3[["Mkt-RF", "SMB", "HML"]].copy()
    elif name == "FF-1-factor":
        risky = twenty_of_25(p25).sub(rf, axis=0)
        out = pd.concat([risky, f3[["Mkt-RF"]]], axis=1)
    elif name == "FF-4-factor":
        risky = twenty_of_25(p25).sub(rf, axis=0)
        out = pd.concat([risky, f3[["Mkt-RF", "SMB", "HML"]], mom[["Mom"]]], axis=1)
    else:
        raise KeyError(f"unknown dataset {name!r}; known: {list(C.DGU_DATASETS)}")
    out = _window(out.dropna(how="any"), start, end)
    expected = C.DGU_DATASETS[name]["N"]
    if out.shape[1] != expected:
        raise ValueError(f"{name}: {out.shape[1]} columns, DGU Table 2 says N = {expected}")
    out.attrs["name"] = name
    out.attrs["market_column"] = "Mkt-RF"
    return out


def build_all(start: str | None = C.DGU_SAMPLE_START, end: str | None = C.DGU_SAMPLE_END,
              cache_dir: str = "data/french") -> dict[str, pd.DataFrame]:
    inputs = load_inputs(cache_dir)
    return {name: build(name, inputs, start, end) for name in C.DGU_DATASETS}

### The rules

`ew` is 1/N. `mv` is Sigma^-1 mu on the window's sample moments, normalised. `min` and `vw` are here because they are one line each; `min` is used in notebook 03 and `vw` is the market check. The risk aversion gamma does not appear because it cancels in the normalisation for every unconstrained rule.

In [ ]:
%%writefile src/bp/rules.py
"""
Portfolio rules from DeMiguel, Garlappi and Uppal (2009), section 1.

Each rule is a function of one estimation window of monthly excess returns,
an array of shape (M, N), and returns the vector of relative weights on the N
risky assets, DGU equation (1): the mean-variance solution x = Sigma^-1 mu / gamma
normalised by the absolute value of its sum, so that the weights sum to +1 or
to -1 and the direction of the position is preserved in the few windows where
the risky assets are net short.

Notation follows the paper: mu and Sigma are the sample moments of the window,
M its length, N the number of assets. The risk aversion gamma cancels in the
normalisation for every unconstrained rule, which is why it does not appear.

Rules in this module, with the DGU abbreviation and section:
  ew       1/N, section 1.1
  mv       sample-based mean-variance, section 1.2
  bs       Bayes-Stein shrinkage, section 1.3.2, Jorion (1986)
  min      minimum variance, section 1.4.1
  vw       the value-weighted market, section 1.4.2 (all weight on the market column)
  mv-c     short-sale-constrained mean-variance, section 1.5
  bs-c     short-sale-constrained Bayes-Stein, section 1.5
  min-c    short-sale-constrained minimum variance, section 1.5
  g-min-c  minimum variance with every weight at least 1/(2N), section 1.5

The constrained rules are quadratic programmes. Each is solved exactly with
the Lawson-Hanson non-negative least squares algorithm (scipy.optimize.nnls)
after a change of variables, so there is no solver setting to tune:
  minimise  x' Sigma x / 2 - mu' x   subject to x >= 0
is, with Sigma = L L' (Cholesky), the same as minimising |L' x - L^-1 mu|^2,
a non-negative least squares problem. A budget constraint 1'w = 1 is added
as one heavily weighted extra row, then the result is renormalised; a lower
bound w >= a is a shift of variables. tests/ checks every one against an
independent solver.
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import nnls

from . import constants as C


def sample_moments(R: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Sample mean and covariance (divisor M - 1) of an (M, N) window."""
    R = np.asarray(R, dtype=float)
    mu = R.mean(axis=0)
    Sigma = np.cov(R, rowvar=False, ddof=1)
    return mu, np.atleast_2d(Sigma)


def normalise(x: np.ndarray) -> np.ndarray:
    """DGU equation (1): w = x / |1'x|."""
    s = float(np.sum(x))
    if s == 0.0:
        raise ZeroDivisionError("portfolio weights sum to zero; cannot normalise")
    return x / abs(s)


def ew(R: np.ndarray) -> np.ndarray:
    n = np.asarray(R).shape[1]
    return np.full(n, 1.0 / n)


def mv(R: np.ndarray) -> np.ndarray:
    mu, Sigma = sample_moments(R)
    x = np.linalg.solve(Sigma, mu)
    return normalise(x)


def min_variance(R: np.ndarray) -> np.ndarray:
    _, Sigma = sample_moments(R)
    ones = np.ones(Sigma.shape[0])
    x = np.linalg.solve(Sigma, ones)
    return x / x.sum()


def vw(market_index: int):
    """The value-weighted market: weight one on the market column, zero elsewhere."""
    def rule(R: np.ndarray) -> np.ndarray:
        w = np.zeros(np.asarray(R).shape[1])
        w[market_index] = 1.0
        return w
    rule.__name__ = "vw"
    return rule


def in_sample_sharpe(R: np.ndarray, rule=mv) -> float:
    """DGU equation (13): the Sharpe ratio of a rule estimated once on the whole sample."""
    mu, Sigma = sample_moments(R)
    w = rule(R)
    return float(w @ mu / np.sqrt(w @ Sigma @ w))


# ---------------------------------------------------------------------------
# Bayes-Stein, DGU section 1.3.2, equations (4) and (5); Jorion (1986)
# ---------------------------------------------------------------------------

def bayes_stein_from_moments(mu: np.ndarray, Sigma_mnm2: np.ndarray, M: int) -> tuple[np.ndarray, np.ndarray, float]:
    """Jorion's predictive moments from a window's mean and its covariance with divisor M - N - 2.

    Returns (mu_bs, Sigma_bs, phi). The mean is shrunk toward the mean return
    of the minimum-variance portfolio by phi (DGU equation 5); the covariance
    is inflated for the uncertainty in the mean, Jorion's predictive variance.
    """
    N = len(mu)
    Sigma_inv_1 = np.linalg.solve(Sigma_mnm2, np.ones(N))
    w_min = Sigma_inv_1 / Sigma_inv_1.sum()
    mu_min = float(mu @ w_min)
    d = mu - mu_min
    dist = float(d @ np.linalg.solve(Sigma_mnm2, d))     # (mu - mu_min 1)' Sigma^-1 (mu - mu_min 1)
    lam = (N + 2) / dist
    phi = lam / (M + lam)                                 # equals (N+2) / ((N+2) + M * dist), DGU equation (5)
    mu_bs = (1.0 - phi) * mu + phi * mu_min
    Sigma_bs = Sigma_mnm2 * (1.0 + 1.0 / (M + lam)) + (lam / (M * (M + 1.0 + lam))) * np.outer(np.ones(N), np.ones(N)) / Sigma_inv_1.sum()
    return mu_bs, Sigma_bs, phi


def bayes_stein_moments(R: np.ndarray) -> tuple[np.ndarray, np.ndarray, float]:
    """The predictive mean and covariance of Jorion (1986), as DGU implement them, from a window of returns.

    The sample covariance here divides by M - N - 2 (DGU equation 5), the
    unbiased estimator of the inverse. Returns (mu_bs, Sigma_bs, phi).
    """
    R = np.asarray(R, dtype=float)
    M, N = R.shape
    mu = R.mean(axis=0)
    dev = R - mu
    Sigma = dev.T @ dev / (M - N - 2)
    return bayes_stein_from_moments(mu, Sigma, M)


def bs(R: np.ndarray) -> np.ndarray:
    mu_bs, Sigma_bs, _ = bayes_stein_moments(R)
    return normalise(np.linalg.solve(Sigma_bs, mu_bs))


# ---------------------------------------------------------------------------
# Constrained rules, DGU section 1.5, solved as non-negative least squares
# ---------------------------------------------------------------------------

_BUDGET_WEIGHT = 1e6   # weight of the budget row; the residual on 1'w = 1 is then below 1e-12 and the result is renormalised anyway


def _cholesky_upper(Sigma: np.ndarray) -> np.ndarray:
    """L' with Sigma = L L'. A tiny ridge keeps a near-singular window solvable."""
    N = Sigma.shape[0]
    try:
        L = np.linalg.cholesky(Sigma)
    except np.linalg.LinAlgError:
        L = np.linalg.cholesky(Sigma + 1e-12 * np.trace(Sigma) / N * np.eye(N))
    return L.T


def nonneg_mean_variance(mu: np.ndarray, Sigma: np.ndarray) -> np.ndarray:
    """argmax mu'x - x'Sigma x / 2 over x >= 0, as NNLS: minimise |L'x - L^-1 mu|^2.

    This is DGU's Lagrangian (8) read literally: the non-negativity constraint
    alone, the scale then removed by normalise(). It is kept so that the two
    readings of the equation can be compared (notebook 03); budget_mean_variance
    is the reading that reproduces the paper's tables.
    """
    Lt = _cholesky_upper(Sigma)
    b = np.linalg.solve(Lt.T, mu)           # L^-1 mu
    x, _ = nnls(Lt, b)
    return x


def budget_mean_variance(mu: np.ndarray, Sigma: np.ndarray, gamma: float = C.DGU_RISK_AVERSION) -> np.ndarray:
    """argmax mu'w - gamma w'Sigma w / 2 over w >= 0 and 1'w = 1.

    The budget constraint sits inside the optimisation, so gamma no longer
    cancels: with gamma = 1 and monthly moments the risk term is small against
    the differences in means, and the rule often puts all wealth in the asset
    with the highest estimated mean. That is the "corner solutions with all
    wealth invested in a single asset" of DGU footnote 22, and it is what
    reproduces their Tables 3 to 5 (constants.DGU_CONSTRAINED_BUDGET_INSIDE).
    As NNLS: minimise |sqrt(gamma) L'w - L^-1 mu / sqrt(gamma)|^2 with the
    budget as a heavily weighted extra row.
    """
    N = len(mu)
    Lt = _cholesky_upper(Sigma)
    A = np.vstack([np.sqrt(gamma) * Lt, _BUDGET_WEIGHT * np.ones(N)])
    b = np.concatenate([np.linalg.solve(Lt.T, mu) / np.sqrt(gamma), [_BUDGET_WEIGHT]])
    w, _ = nnls(A, b)
    return w / w.sum()


def constrained_min_variance(Sigma: np.ndarray, lower: float = 0.0) -> np.ndarray:
    """argmin w'Sigma w over w >= lower, 1'w = 1, as NNLS on v = w - lower.

    Objective |L'(v + lower 1)|^2, so the NNLS target is -L' lower 1; the budget
    1'v = 1 - N lower is appended as a heavily weighted row.
    """
    N = Sigma.shape[0]
    Lt = _cholesky_upper(Sigma)
    ones = np.ones(N)
    A = np.vstack([Lt, _BUDGET_WEIGHT * ones])
    b = np.concatenate([-Lt @ (lower * ones), [_BUDGET_WEIGHT * (1.0 - N * lower)]])
    v, _ = nnls(A, b)
    w = v + lower
    return w / w.sum()


def mv_c(R: np.ndarray) -> np.ndarray:
    mu, Sigma = sample_moments(R)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu, Sigma)
    return normalise(nonneg_mean_variance(mu, Sigma))


def bs_c(R: np.ndarray) -> np.ndarray:
    mu_bs, Sigma_bs, _ = bayes_stein_moments(R)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu_bs, Sigma_bs)
    return normalise(nonneg_mean_variance(mu_bs, Sigma_bs))


def mv_c_cone(R: np.ndarray) -> np.ndarray:
    """Equation (8) read literally, for the comparison in notebook 03; not a reported rule."""
    mu, Sigma = sample_moments(R)
    return normalise(nonneg_mean_variance(mu, Sigma))


def min_c(R: np.ndarray) -> np.ndarray:
    _, Sigma = sample_moments(R)
    return constrained_min_variance(Sigma, lower=0.0)


def g_min_c(R: np.ndarray) -> np.ndarray:
    """DGU p. 1926: minimum variance with w >= a 1, a = 1/(2N), halfway between min-c and 1/N."""
    _, Sigma = sample_moments(R)
    N = Sigma.shape[0]
    return constrained_min_variance(Sigma, lower=C.DGU_GMINC_LOWER_BOUND_FRACTION / N)


# ---------------------------------------------------------------------------
# The same rules from a window's moments, for the long simulated histories of
# notebook 04, where recomputing the covariance at every month would dominate
# the running time. Each takes (mu, Sigma, M) with Sigma the sample covariance
# with divisor M - 1, and returns the same weights as its window-based twin
# (tests/test_rules_backtest.py checks the equality).
# ---------------------------------------------------------------------------

def _to_mnm2(Sigma: np.ndarray, M: int) -> np.ndarray:
    """Rescale a divisor-(M-1) covariance to divisor M - N - 2, DGU equation (5)."""
    N = Sigma.shape[0]
    return Sigma * (M - 1) / (M - N - 2)


def ew_m(mu, Sigma, M):
    return np.full(len(mu), 1.0 / len(mu))


def mv_m(mu, Sigma, M):
    return normalise(np.linalg.solve(Sigma, mu))


def min_m(mu, Sigma, M):
    x = np.linalg.solve(Sigma, np.ones(len(mu)))
    return x / x.sum()


def bs_m(mu, Sigma, M):
    mu_bs, Sigma_bs, _ = bayes_stein_from_moments(mu, _to_mnm2(Sigma, M), M)
    return normalise(np.linalg.solve(Sigma_bs, mu_bs))


def mv_c_m(mu, Sigma, M):
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu, Sigma)
    return normalise(nonneg_mean_variance(mu, Sigma))


def bs_c_m(mu, Sigma, M):
    mu_bs, Sigma_bs, _ = bayes_stein_from_moments(mu, _to_mnm2(Sigma, M), M)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu_bs, Sigma_bs)
    return normalise(nonneg_mean_variance(mu_bs, Sigma_bs))


def min_c_m(mu, Sigma, M):
    return constrained_min_variance(Sigma, lower=0.0)


def g_min_c_m(mu, Sigma, M):
    return constrained_min_variance(Sigma, lower=C.DGU_GMINC_LOWER_BOUND_FRACTION / len(mu))


MOMENT_RULES = {
    "ew": ew_m,
    "mv": mv_m,
    "bs": bs_m,
    "min": min_m,
    "mv-c": mv_c_m,
    "bs-c": bs_c_m,
    "min-c": min_c_m,
    "g-min-c": g_min_c_m,
}


# The registry. vw needs the market column and is built per dataset with vw(index).
RULES = {
    "ew": ew,
    "mv": mv,
    "bs": bs,
    "min": min_variance,
    "mv-c": mv_c,
    "bs-c": bs_c,
    "min-c": min_c,
    "g-min-c": g_min_c,
}

### The rolling evaluation

`rolling` walks the window forward one month at a time and records, for each out-of-sample month, the return earned, the target weights, the weights the month's returns drifted the previous portfolio to, and the trade between the two. The measures follow DGU's equations; the two p-values are the Jobson-Korkie test for Sharpe ratios and the delta-method test for CEQ returns (a standard way to obtain a standard error for a quantity built from means and variances), both against 1/N.

In [ ]:
%%writefile src/bp/backtest.py
"""
Rolling out-of-sample evaluation and the three performance measures of
DeMiguel, Garlappi and Uppal (2009), section 2.

The rolling sample: with T months of data and an estimation window of M
months, the rule is estimated on months t-M+1 to t and its weights are held
through month t+1, for t = M, ..., T-1. That gives T - M out-of-sample
monthly returns per rule (DGU p. 1928).

Measures, all monthly:
  Sharpe ratio        mean of out-of-sample excess returns over their standard
                      deviation, equation (12); the p-value of the difference
                      from 1/N is the Jobson-Korkie (1981) statistic with the
                      Memmel (2003) correction, footnote 16.
  CEQ return          mean minus gamma/2 times variance, equation (14), with
                      the delta-method p-value of footnote 18.
  turnover            average over months of the sum of absolute trades needed
                      to move from the drifted weights to the new target,
                      equation (15).
  net-of-cost return  equation (16), with a proportional cost c per unit of
                      one-way turnover.
  return-loss         equation (17): the extra mean return a rule needs for its
                      net Sharpe ratio to equal 1/N's.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy import stats

from . import constants as C


@dataclass
class Backtest:
    name: str
    oos: pd.Series            # out-of-sample excess return, indexed by the month it was earned
    weights: pd.DataFrame     # target weights, indexed by the month they were formed for
    turnover: pd.Series       # one-way trade at each rebalance, equation (15) summand
    drifted: pd.DataFrame     # weights before each rebalance


def drift(w: np.ndarray, r: np.ndarray) -> np.ndarray:
    """Weights after one month of excess returns r, before rebalancing (DGU p. 1929, w_{t+}).

    Position j grows to w_j (1 + r_j) and wealth to 1 + w'r, the residual
    1 - sum(w) sitting in the riskless asset at zero excess return. Dividing by
    the sum of the risky positions would lose the sign of a net-short portfolio;
    dividing by wealth keeps it, as DGU's normalisation does on purpose.
    """
    return w * (1.0 + r) / (1.0 + w @ r)


def rolling(returns: pd.DataFrame, rule, window: int = C.DGU_ESTIMATION_WINDOW, name: str | None = None) -> Backtest:
    R = returns.to_numpy(dtype=float)
    T, N = R.shape
    if T <= window:
        raise ValueError(f"{T} months is not more than the {window}-month window")
    idx = returns.index
    n_oos = T - window
    oos = np.empty(n_oos)
    weights = np.empty((n_oos, N))
    drifted = np.full((n_oos, N), np.nan)
    turnover = np.full(n_oos, np.nan)
    prev = None
    for k in range(n_oos):
        t = window + k                       # the month whose return is earned out of sample
        w = np.asarray(rule(R[t - window:t]), dtype=float)
        weights[k] = w
        if prev is not None:
            drifted[k] = prev
            turnover[k] = np.abs(w - prev).sum()
        oos[k] = w @ R[t]
        prev = drift(w, R[t])
    months = idx[window:]
    cols = list(returns.columns)
    return Backtest(
        name=name or getattr(rule, "__name__", "rule"),
        oos=pd.Series(oos, index=months, name=name),
        weights=pd.DataFrame(weights, index=months, columns=cols),
        turnover=pd.Series(turnover, index=months, name="turnover"),
        drifted=pd.DataFrame(drifted, index=months, columns=cols),
    )


def rolling_moments(returns: pd.DataFrame | np.ndarray, rules: dict, window: int = C.DGU_ESTIMATION_WINDOW,
                    refresh_every: int = 1000, progress: int = 0) -> dict[str, Backtest]:
    """Rolling evaluation of a set of moment-based rules at once, for long simulated histories.

    The window's mean and covariance are kept as running sums, updated in
    O(N^2) per month (one row in, one row out) instead of recomputed from the
    whole window, and recomputed from scratch every refresh_every months so
    floating-point drift cannot accumulate. Every rule in `rules` is a
    function (mu, Sigma, M) -> weights, as in rules.MOMENT_RULES, and
    sees exactly the moments the window-based rules would compute (Sigma
    with divisor M - 1). Returns one Backtest per rule.
    """
    if isinstance(returns, pd.DataFrame):
        R = returns.to_numpy(dtype=float)
        idx = returns.index
        cols = list(returns.columns)
    else:
        R = np.asarray(returns, dtype=float)
        idx = pd.RangeIndex(len(R))
        cols = list(range(R.shape[1]))
    T, N = R.shape
    if T <= window:
        raise ValueError(f"{T} months is not more than the {window}-month window")
    n_oos = T - window
    names = list(rules)
    oos = {k: np.empty(n_oos) for k in names}
    weights = {k: np.empty((n_oos, N)) for k in names}
    drifted = {k: np.full((n_oos, N), np.nan) for k in names}
    turnover = {k: np.full(n_oos, np.nan) for k in names}
    prev = {k: None for k in names}
    s1 = R[:window].sum(axis=0)
    s2 = R[:window].T @ R[:window]
    for k_step in range(n_oos):
        t = window + k_step
        if k_step > 0 and k_step % refresh_every == 0:
            s1 = R[t - window:t].sum(axis=0)
            s2 = R[t - window:t].T @ R[t - window:t]
        mu = s1 / window
        Sigma = (s2 - window * np.outer(mu, mu)) / (window - 1)
        Sigma = 0.5 * (Sigma + Sigma.T)
        r_next = R[t]
        for k in names:
            w = np.asarray(rules[k](mu, Sigma, window), dtype=float)
            weights[k][k_step] = w
            if prev[k] is not None:
                drifted[k][k_step] = prev[k]
                turnover[k][k_step] = np.abs(w - prev[k]).sum()
            oos[k][k_step] = w @ r_next
            prev[k] = drift(w, r_next)
        # slide the window: drop month t - window, add month t
        r_out = R[t - window]
        s1 = s1 - r_out + r_next
        s2 = s2 - np.outer(r_out, r_out) + np.outer(r_next, r_next)
        if progress and (k_step + 1) % progress == 0:
            print(f"  {k_step + 1}/{n_oos} months", flush=True)
    months = idx[window:]
    return {k: Backtest(name=k, oos=pd.Series(oos[k], index=months, name=k),
                        weights=pd.DataFrame(weights[k], index=months, columns=cols),
                        turnover=pd.Series(turnover[k], index=months, name="turnover"),
                        drifted=pd.DataFrame(drifted[k], index=months, columns=cols)) for k in names}


def subperiod(bt: Backtest, start=None, end=None) -> Backtest:
    """The same rolling evaluation restricted to the out-of-sample months from start to end (Periods or
    'YYYY-MM' strings, inclusive). Every rebalance that falls inside the period counts towards its turnover."""
    idx = bt.oos.index
    lo = pd.Period(start, "M") if isinstance(start, str) else (start if start is not None else idx[0])
    hi = pd.Period(end, "M") if isinstance(end, str) else (end if end is not None else idx[-1])
    sl = slice(lo, hi)
    return Backtest(name=bt.name, oos=bt.oos.loc[sl], weights=bt.weights.loc[sl],
                    turnover=bt.turnover.loc[sl], drifted=bt.drifted.loc[sl])


# ---------------------------------------------------------------------------
# Measures
# ---------------------------------------------------------------------------

def sharpe(x: pd.Series) -> float:
    return float(x.mean() / x.std(ddof=1))


def ceq(x: pd.Series, gamma: float = C.DGU_RISK_AVERSION) -> float:
    return float(x.mean() - 0.5 * gamma * x.var(ddof=1))


def mean_turnover(bt: Backtest) -> float:
    """Equation (15): the average one-way trade per rebalance, over the rebalances that exist."""
    return float(bt.turnover.dropna().mean())


def net_of_cost(bt: Backtest, cost: float = C.DGU_TCOST) -> pd.Series:
    """Equation (16): (1 + r)(1 - c * turnover) - 1, with no trade charged in the first month."""
    to = bt.turnover.fillna(0.0)
    return (1.0 + bt.oos) * (1.0 - cost * to) - 1.0


def return_loss(bt: Backtest, benchmark: Backtest, cost: float = C.DGU_TCOST) -> float:
    """Equation (17), on net-of-cost returns."""
    k = net_of_cost(bt, cost)
    b = net_of_cost(benchmark, cost)
    return float(b.mean() / b.std(ddof=1) * k.std(ddof=1) - k.mean())


def sharpe_pvalue(x: pd.Series, y: pd.Series) -> float:
    """Jobson-Korkie (1981) test of equal Sharpe ratios with the Memmel (2003) correction, DGU footnote 16."""
    T = len(x)
    mi, mn = x.mean(), y.mean()
    si, sn = x.std(ddof=1), y.std(ddof=1)
    s_in = np.cov(x, y, ddof=1)[0, 1]
    theta = (2 * si**2 * sn**2 - 2 * si * sn * s_in
             + 0.5 * mi**2 * sn**2 + 0.5 * mn**2 * si**2
             - (mi * mn / (si * sn)) * s_in**2) / T
    z = (sn * mi - si * mn) / np.sqrt(theta)
    return float(2 * (1 - stats.norm.cdf(abs(z))))


def ceq_pvalue(x: pd.Series, y: pd.Series, gamma: float = C.DGU_RISK_AVERSION) -> float:
    """Delta-method test of equal CEQ returns, DGU footnote 18."""
    T = len(x)
    mi, mn = x.mean(), y.mean()
    vi, vn = x.var(ddof=1), y.var(ddof=1)
    v_in = np.cov(x, y, ddof=1)[0, 1]
    grad = np.array([1.0, -1.0, -0.5 * gamma, 0.5 * gamma])
    Theta = np.array([
        [vi, v_in, 0.0, 0.0],
        [v_in, vn, 0.0, 0.0],
        [0.0, 0.0, 2 * vi**2, 2 * v_in**2],
        [0.0, 0.0, 2 * v_in**2, 2 * vn**2],
    ])
    diff = (mi - 0.5 * gamma * vi) - (mn - 0.5 * gamma * vn)
    var = grad @ Theta @ grad / T
    z = diff / np.sqrt(var)
    return float(2 * (1 - stats.norm.cdf(abs(z))))


def evaluate(bts: dict[str, Backtest], benchmark: str = "ew", cost: float = C.DGU_TCOST) -> pd.DataFrame:
    """One row per rule: Sharpe, its p-value against the benchmark, CEQ, its p-value, turnover, relative turnover, return-loss."""
    b = bts[benchmark]
    rows = []
    for name, bt in bts.items():
        rows.append({
            "rule": name,
            "sharpe": sharpe(bt.oos),
            "sharpe_p": np.nan if name == benchmark else sharpe_pvalue(bt.oos, b.oos),
            "ceq": ceq(bt.oos),
            "ceq_p": np.nan if name == benchmark else ceq_pvalue(bt.oos, b.oos),
            "turnover": mean_turnover(bt),
            "turnover_rel": mean_turnover(bt) / mean_turnover(b),
            "return_loss": np.nan if name == benchmark else return_loss(bt, b, cost),
        })
    return pd.DataFrame(rows).set_index("rule")

### The comparison

Replicated beside published, one row per rule and dataset, with the verdict of the pre-committed band where one applies.

In [ ]:
%%writefile src/bp/compare.py
"""
Replicated numbers beside published ones, with the pre-committed band
applied where one exists.

Every table has one row per (rule, dataset): the replicated value, DGU's
value from src/bp/dgu_published.py, the difference, and a verdict column.
The verdict is "pass" or "MISS" where a band applies (Sharpe ratios,
turnover below the relative level in constants), "report" where the number
is shown beside the published one with no pass or fail (CEQ, return-loss, the
explosive unconstrained turnovers), and "check" for rows with no published counterpart.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import dgu_published as P

DATASETS = list(P.DATASETS)


def _published(table: dict, rule: str, dataset: str):
    entry = table.get(rule)
    if entry is None:
        return np.nan
    vals = entry[0] if isinstance(entry, tuple) and isinstance(entry[0], tuple) else entry
    return vals[DATASETS.index(dataset)]


def sharpe_table(results: dict[str, pd.DataFrame], in_sample: dict[str, float] | None = None) -> pd.DataFrame:
    """results: dataset -> evaluate() frame. in_sample: dataset -> in-sample mv Sharpe ratio."""
    rows = []
    for dataset, frame in results.items():
        if in_sample is not None and dataset in in_sample:
            pub = _published(P.SHARPE, "mv (in sample)", dataset)
            rep = in_sample[dataset]
            rows.append({"rule": "mv (in sample)", "dataset": dataset, "replicated": rep, "published": pub,
                         "difference": rep - pub, "verdict": _sharpe_verdict("mv (in sample)", dataset, rep - pub)})
        for rule in frame.index:
            pub = _published(P.SHARPE, rule, dataset)
            rep = frame.loc[rule, "sharpe"]
            rows.append({"rule": rule, "dataset": dataset, "replicated": rep, "published": pub,
                         "difference": rep - pub, "verdict": _sharpe_verdict(rule, dataset, rep - pub)})
    return pd.DataFrame(rows).set_index(["rule", "dataset"])


def sharpe_is_gated(rule: str, dataset: str) -> bool:
    """The band applies to out-of-sample rows whose published relative turnover is below the level in constants."""
    if rule == "mv (in sample)":
        return False
    if rule == "ew":
        return True
    rel = _published(P.TURNOVER_RELATIVE, rule, dataset)
    if np.isnan(rel):
        return True
    return rel < C.BAND_SHARPE_APPLIES_BELOW_RELATIVE


def _sharpe_verdict(rule: str, dataset: str, diff: float) -> str:
    if np.isnan(diff):
        return "check"
    if not sharpe_is_gated(rule, dataset):
        return "report"
    return "pass" if abs(diff) <= C.BAND_SHARPE_ABS else "MISS"


def ceq_table(results: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for dataset, frame in results.items():
        for rule in frame.index:
            pub = _published(P.CEQ, rule, dataset)
            rep = frame.loc[rule, "ceq"]
            rows.append({"rule": rule, "dataset": dataset, "replicated": rep, "published": pub,
                         "difference": rep - pub, "verdict": "report" if not np.isnan(pub) else "check"})
    return pd.DataFrame(rows).set_index(["rule", "dataset"])


def turnover_table(results: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """1/N's absolute turnover against Table 5's first line; every other rule relative to 1/N against panel A."""
    rows = []
    for dataset, frame in results.items():
        j = DATASETS.index(dataset)
        for rule in frame.index:
            if rule == "ew":
                rep, pub = frame.loc[rule, "turnover"], P.TURNOVER_1N[j]
            else:
                rep, pub = frame.loc[rule, "turnover_rel"], _published(P.TURNOVER_RELATIVE, rule, dataset)
            if np.isnan(pub):
                verdict = "check"
            elif rule != "ew" and pub >= C.BAND_TURNOVER_APPLIES_BELOW_RELATIVE:
                verdict = "report"
            elif pub == 0.0:
                verdict = "pass" if rep == 0.0 else "MISS"
            else:
                verdict = "pass" if abs(rep / pub - 1.0) <= C.BAND_TURNOVER_REL else "MISS"
            rows.append({"rule": rule, "dataset": dataset, "replicated": rep, "published": pub,
                         "relative_error": np.nan if (np.isnan(pub) or pub == 0.0) else rep / pub - 1.0,
                         "verdict": verdict})
    return pd.DataFrame(rows).set_index(["rule", "dataset"])


def return_loss_table(results: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for dataset, frame in results.items():
        for rule in frame.index:
            if rule == "ew":
                continue
            pub = _published(P.RETURN_LOSS, rule, dataset)
            rep = frame.loc[rule, "return_loss"]
            rows.append({"rule": rule, "dataset": dataset, "replicated": rep, "published": pub,
                         "difference": rep - pub, "verdict": "report" if not np.isnan(pub) else "check"})
    return pd.DataFrame(rows).set_index(["rule", "dataset"])


def top3_preserved(results: dict[str, pd.DataFrame], rules: list[str]) -> pd.DataFrame:
    """Per dataset: the top three rules by replicated Sharpe ratio, by published, and whether the order matches."""
    rows = []
    for dataset, frame in results.items():
        gated = [r for r in rules if sharpe_is_gated(r, dataset)]
        rep = frame.loc[gated, "sharpe"].sort_values(ascending=False).index[:3].tolist()
        pub = pd.Series({r: _published(P.SHARPE, r, dataset) for r in gated}).sort_values(ascending=False).index[:3].tolist()
        rows.append({"dataset": dataset, "rules_ranked": gated, "replicated_top3": rep, "published_top3": pub,
                     "verdict": "pass" if rep == pub else "MISS"})
    return pd.DataFrame(rows).set_index("dataset")

## Build the datasets

Each is monthly excess returns over French's one-month Treasury bill, 1963-07 to 2004-11, with the factor portfolios as the last columns. The 20 size-and-value portfolios are the 25 less the five with the largest firms, DGU footnote 24.

In [ ]:
import sys
sys.path.insert(0, "src")
import numpy as np
import pandas as pd
from bp import constants as C
from bp import datasets as D
from bp import rules as S
from bp import backtest as B
from bp import compare as CMP
from bp import dgu_published as P

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

data = D.build_all()
for name, R in data.items():
    assert R.shape == (C.DGU_EXPECTED_T, C.DGU_DATASETS[name]["N"]), (name, R.shape)
    print(f"{name:12s} {R.shape[0]} months x {R.shape[1]} assets, {R.index[0]} to {R.index[-1]}, columns end with {list(R.columns[-2:])}")

## Run the two rules, and the market

377 out-of-sample months per rule and dataset. The in-sample mean-variance Sharpe ratio, estimated once on all 497 months, is computed beside them: it is the upper bound, the Sharpe ratio a mean-variance investor would earn with no estimation error.

In [ ]:
results, in_sample, runs = {}, {}, {}
for name, R in data.items():
    mkt = list(R.columns).index("Mkt-RF")
    rules = {"ew": S.ew, "mv": S.mv, "vw": S.vw(mkt)}
    bts = {k: B.rolling(R, f, name=k) for k, f in rules.items()}
    assert all(len(bt.oos) == C.DGU_EXPECTED_OOS for bt in bts.values())
    runs[name] = bts
    results[name] = B.evaluate(bts)
    in_sample[name] = S.in_sample_sharpe(R.to_numpy())
    print(f"{name:12s} done: {len(bts['ew'].oos)} out-of-sample months, {bts['ew'].oos.index[0]} to {bts['ew'].oos.index[-1]}")

## Table 3: Sharpe ratios

In [ ]:
sharpe_cmp = CMP.sharpe_table(results, in_sample)
sharpe_cmp

## Table 4: certainty-equivalent returns

No band applies to the CEQ; the rows are reported beside the published ones. For the mean-variance rule the CEQ is dominated by the variance term, and the variance is dominated by the months with extreme weights (the next table shows the five largest), so its value is a statement about those months.

In [ ]:
CMP.ceq_table(results)

## Table 5: turnover and return-loss

1/N's turnover is its absolute monthly figure; the other rows are relative to 1/N. Return-loss is the extra monthly return a rule would need for its Sharpe ratio net of a 50 basis point cost to equal 1/N's, equation (17).

In [ ]:
CMP.turnover_table(results)

In [ ]:
CMP.return_loss_table(results)

## Where the mean-variance rule goes wrong, month by month

The five out-of-sample months with the largest absolute mean-variance return on the Industry dataset, with the sum of absolute weights the rule held going into each. A sum of absolute weights of 50 means the rule was 2,500% long and 2,400% short in some combination of eleven industry portfolios, on the strength of ten years of monthly data.

In [ ]:
mv_ind = runs["Industry"]["mv"]
worst = mv_ind.oos.abs().sort_values(ascending=False).head(5).index
pd.DataFrame({
    "mv return": mv_ind.oos.loc[worst],
    "sum |weights|": mv_ind.weights.loc[worst].abs().sum(axis=1),
    "1/N return": runs["Industry"]["ew"].oos.loc[worst],
})

## The fragility of the mean-variance row

This test separates a miss caused by the code from a miss caused by the data. Add Gaussian noise to every return, at one basis point and at ten (a tenth of a percent, small against monthly standard deviations of four to six percent and below the size of many revisions French makes to a published month), and rerun the rule fifty times at each level with a fixed seed. If the published number sits inside the range the noise produces, no code can replicate the row to a tighter band on any vintage. The known-answer test in `tests/test_rules_backtest.py`, which recovers the tangency weights from simulated data, shows that the code is right. This cell measures how far the data can move the number. About two minutes.

In [ ]:
rng = np.random.default_rng(C.SIM_SEED)
rows = []
for name, R in data.items():
    j = CMP.DATASETS.index(name)
    row = {"dataset": name, "mv OOS": results[name].loc["mv", "sharpe"], "OOS published": P.SHARPE["mv"][0][j],
           "in-sample": in_sample[name], "in-sample published": P.SHARPE["mv (in sample)"][0][j]}
    for bp in (1, 10):
        oos, ins = [], []
        for _ in range(50):
            Rp = R + rng.normal(0.0, bp * 1e-4, size=R.shape)
            oos.append(B.sharpe(B.rolling(Rp, S.mv).oos))
            ins.append(S.in_sample_sharpe(Rp.to_numpy()))
        row[f"OOS min {bp}bp"], row[f"OOS max {bp}bp"] = min(oos), max(oos)
        row[f"IS min {bp}bp"], row[f"IS max {bp}bp"] = min(ins), max(ins)
    rows.append(row)
fragility = pd.DataFrame(rows).set_index("dataset")
fragility[["mv OOS", "OOS published", "OOS min 1bp", "OOS max 1bp", "OOS min 10bp", "OOS max 10bp"]]

In [ ]:
fragility[["in-sample", "in-sample published", "IS min 1bp", "IS max 1bp", "IS min 10bp", "IS max 10bp"]]

## What this notebook established, and what could be wrong

On the vintage of 10 September 2026 the 1/N rows replicate to the third decimal on three datasets (0.1353 against 0.1353 on Industry) and to 0.011 on MKT/SMB/HML, whose three assets are the factors themselves and have been revised more than the portfolios. The market row is within 0.0025 everywhere. 1/N's turnover is within 2.5% of the paper on every dataset, which fixes the turnover convention: trades are measured against the weights the portfolio has drifted to after the month's returns (an asset that rose became a larger share of the portfolio, and the trade back to 1/N is the turnover), and the first purchase is not counted.

The unconstrained mean-variance rule does not replicate within the 0.03 band on Industry or FF-1-factor, and the fragility table says why it cannot: ten basis points of noise move its out-of-sample Sharpe ratio across a range of 0.14 on Industry, 0.12 on FF-1-factor and 0.25 on FF-4-factor, and the published figure lies inside that range on all three. One basis point is already enough on the two size-and-value datasets. In each of the three datasets the market return is nearly a weighted average of the other assets, so the rule sees two assets that move almost identically. The covariance matrix then has almost no information about the difference between them, and inverting it divides by a number close to zero: a tiny change in the inputs becomes a huge change in the weights, and the rule can hold an enormous long position in one and an enormous short position in the other at almost no measured risk. The January 1975 position of 4,500 times wealth is an example. On MKT/SMB/HML, three assets with no such near-duplicate among them, the same rule moves by 0.001 under the noise and passes. The band as first written did not anticipate a row that no vintage can reproduce. The amendment at the top of this notebook follows from this table.

The in-sample mean-variance Sharpe ratios on the two size-and-value datasets are 0.04 to 0.06 below the paper's, and those two numbers move by less than 0.01 under ten basis points of noise, so the difference is in the data. Two causes are possible. The first is revisions to French's 25 portfolios since the paper's 2004 download. The second is the risk-free rate, the Treasury bill return that is subtracted from each portfolio's return to make it an excess return: DGU describe a 90-day bill in one place and a one-month bill in another, and French's site carries only the one-month series, which this notebook uses. Shifting the excess returns of the long-only assets by 3 basis points (0.03 percentage points) a month moves the FF-1-factor in-sample figure by 0.004, far less than the gap of 0.04 to 0.06, so a difference between the two bill rates of that size does not explain the gap. The cause stays unresolved in this notebook.